# 🎫 00 — The standing right

*Why "agents paying each other" is nearly a solved problem — and why the thing this
project sells breaks the solution.*

Welcome to the course. Over eleven notebooks you will rebuild this entire project from
zero — no blockchain background, no networking experience, no Python-typing lore
assumed — until you understand it well enough to **write the paper about it**. That last
part is the point: this is not a tour of a codebase, it is the construction of an
argument you will later defend in front of reviewers.

**The method — rebuild, then reveal.** Reading production code first is the wrong order:
you meet the edge cases before the problem. So every notebook here does the opposite:

1. **A problem, told as a story** — always the same story, with the same numbers
   (you'll meet them in a minute).
2. **Rebuild the solution from scratch**, in plain Python, cell by cell — and the honest
   way: every field and every `if` gets added *because you just watched its absence get
   exploited*. Where the topic is a skill (deploying an AI agent), this step is a genuine
   from-zero tutorial.
3. **Reveal — and run — the real thing**: the project's actual component, on the same
   story, behaving exactly like the toy you just built.

**Three recurring blocks**, introduced here once and used everywhere:

- **🧭 Decision boxes** appear at the exact moment a design choice is made. Each names
  the alternatives that were genuinely on the table and says where the point lands in
  the paper — in one of two honest registers. A **principled** decision is argued: the
  alternative fails for a reason, here it is. A **pragmatic** decision is *not*
  oversold: other options exist and may be better in production; we took the simplest
  thing that demonstrates the mechanism, and the paper admits that in its Limitations
  section rather than dressing it up as a design argument.
- **📝 For the paper**, each notebook's closing section: draft sentences you could
  nearly paste into the paper, each paired with the evidence *you personally ran*, plus
  the reviewer objections you can now answer.
- **✏️ Your turn** exercises, embedded where each concept lands: a scaffold cell you
  edit, a prediction you write down *before* running, and a fold-out solution below.
  Try first, peek second.

**You need:** nothing. This opening chapter is pure Python — no blockchain, no router,
no AI model. Run every cell, in order.

## 1 · The story (and the numbers you'll see two hundred times)

Meet **Ada**. Ada is an AI agent — a program with a language model for a brain and a
crypto wallet for a hand, acting on her owner's behalf. At 13:30 on a Tuesday her
owner's application hands her a job: move a **45 GB dataset** from `host-A` to
`host-B`, finished **before 16:00**.

Ada does the arithmetic every network engineer does first:

In [ ]:
dataset_gigabytes = 45
window_hours = 2

bits_to_move = dataset_gigabytes * 8 * 10**9        # 1 byte = 8 bits
seconds_available = window_hours * 3600

rate_needed = bits_to_move / seconds_available       # bits per second
print(f"{rate_needed / 1e6:.0f} Mbps, steadily, for the whole two hours")

The regular best-effort network *might* give her 50 Mbps, or might not — and "might" is
not a plan when there's a deadline. Ada needs a **guarantee**: 50 megabits per second,
on the path from A to B, from 14:00 to 16:00.

Meet **Bell**. Bell is also an AI agent. Bell works for a network operator — his owner
runs real routers — and Bell sells exactly this: guaranteed bandwidth, by the hour.
Price for Ada's window: **10 TOK** (a token — think "casino chip": a unit of value that
lives in a shared ledger both parties can see; chapter 01 builds one).

In today's world, what happens between Ada and Bell involves humans: a sales call, a
contract, an account, an API key arriving by email on Thursday. Ada's deadline is in two
and a half hours. What this project wants instead: **Ada buys the guarantee from Bell,
by herself, in seconds — though they have never met and have no reason to trust each
other. And then the network actually obeys.**

That sentence crosses **four trust domains** — four zones with different owners, where
different parties get to lie:

| domain | who controls it | what could go wrong there |
|---|---|---|
| **Ada's world** (her agent + her key) | Ada's owner | Ada might not pay, or might deny she did |
| **Bell's world** (his agent, his gatekeeper) | Bell's owner | Bell might take the money and deliver nothing |
| **the neutral ground** (a shared ledger neither controls) | neither — that's the point | must exist at all; chapters 01 and 03 build it |
| **the network** (physical routers) | Bell's owner, physically | a purchase is just data — someone must turn it into configuration, correctly, and *only* when authorized |

Every notebook in this course is about moving something across one of those borders
without trust. And one discipline, adopted now: **one deal, everywhere.** Ada, Bell,
50 Mbps, 14:00–16:00, 10 TOK, and — once it exists — ticket **#7**: the same canonical
numbers run through the story docs, the tests, the notebooks, and the paper. When you
see a different number, it's a bug.

## 2 · What agents can already buy (and it works!)

Machine-to-machine payment is not science fiction — a real standard exists for it (the
paper calls this family *x402-style*): attach a small payment to an HTTP request, get
the response back, done. Let's build the whole model in fifteen lines, because it's
genuinely simple and genuinely good:

In [ ]:
# A minimal ledger: who holds how many TOK. (Chapter 01 asks what makes one trustworthy.)
balances = {"Ada": 100, "Provider": 0}

def pay(sender, receiver, amount):
    if balances[sender] < amount:
        raise ValueError(f"{sender} only has {balances[sender]} TOK")
    balances[sender] -= amount
    balances[receiver] += amount

def summarize(text):                      # a stand-in for any paid API: work in, result out
    return text[:47] + "..."

def pay_per_call(service, request, price, buyer):
    """The pay-per-call model: payment settles, response comes back, transaction over."""
    pay(buyer, "Provider", price)
    response = service(request)
    receipt = {"paid": price, "by": buyer, "for": "one call"}
    return response, receipt

response, receipt = pay_per_call(summarize, "The vending machine model of settlement replaces trusted intermediaries with...", price=1, buyer="Ada")
print("response:", response)
print("receipt :", receipt)
print("balances:", balances)

Notice why this model is *right* for API work: by the time the money has moved, **the
thing bought has already fully happened**. The summary is in Ada's hands. There is
nothing left to enforce, so a record that "value moved" is all the paperwork anyone
needs.

Now buy Ada's actual deal with the same machinery. The "service" here can only do what
any instantaneous call can do — *promise*:

In [ ]:
def sell_bandwidth(request):
    # An HTTP call cannot deliver two future hours of bandwidth in its response body.
    # The best it can return, at 13:45, is a promise:
    return f"OK — reserved: {request}"

response, receipt = pay_per_call(sell_bandwidth, "50 Mbps, host-A→host-B, 14:00–16:00",
                                 price=10, buyer="Ada")
print("it is 13:45.")
print("response:", response)
print("receipt :", receipt)

The call *succeeded*. Money moved, a polite response came back — and yet Ada bought
nothing she can hold. It is 13:45: **everything she paid for still lies ahead.** The
window hasn't opened; the routers aren't configured; Bell could deliver, half-deliver,
or vanish. Interrogate the only artifact the model left behind:

In [ ]:
the_record = receipt      # all that persists after a pay-per-call purchase
print("the record:", the_record)
print()

questions = [
    "does it name what was promised (50 Mbps, which path)?",
    "does it name the window (14:00–16:00)?",
    "can Bell's gatekeeper consult it at 15:00 to decide 'still valid?'",
    "can it be revoked, or checked for revocation?",
    "does it even say who currently holds the right?",
]
for q in questions:
    print("✗", q)

The record proves *money moved*. It names no terms, no window, no revocation state, no
holder. It is a **receipt** — and a receipt cannot be the thing that was sold.

Here is the sentence this whole project hangs on. An API call is consumed in the
response. A network service is the opposite kind of object: **when payment settles,
everything the buyer paid for still lies ahead** — a claim on the seller's equipment
that must be *honored* for an agreed window, *withdrawn* at expiry, and *revocable*
before it. Call that a **standing right**. Selling a standing right means the sale must
leave behind an object that outlives it: something the seller's own gatekeeper can keep
consulting — at 14:00, at 15:00, at 15:59 — to decide whether the equipment should
still obey.

> **🧭 Decision (principled) — sell an object that outlives the settlement**
>
> **Chosen:** the sale produces a durable, consultable object (the *entitlement* — built
> in chapter 03) carrying terms, window, holder, and revocation state.
> **Alternatives:** (a) per-request payment plus a receipt, with delivery left to
> application logic; (b) the classic subscription: pay, then receive an API key that
> *actually* opens the door.
> **Why they fail:** (a) you just ran — the receipt answers none of the questions a
> gatekeeper must ask for the next two hours. (b) smuggles every trust problem back in
> wearing the key's face: who guarantees the key arrives, arrives only to Ada, can't be
> copied, can't be silently revoked? You'd need a second trust machine to guard the
> first one's receipt — infinite regress, and autonomy dies at the first human step
> (the email with the key).
> **In the paper:** this is §3.1's problem definition, nearly verbatim, and the framing
> of §1's second paragraph.

**✏️ Your turn 1 — the refund that can't see the future**

An obvious patch to pay-per-call: *refund if the service fails*. Below, extend
`pay_per_call` so that if the service returns `None`, the buyer is refunded. Then run it
on the bandwidth purchase and answer in a comment: **does the refund protect Ada?
Predict before you run.**

In [ ]:
def pay_per_call_v2(service, request, price, buyer):
    pay(buyer, "Provider", price)
    response = service(request)
    # TODO: if response is None, refund the buyer (one pay(...) call), return (None, None)
    receipt = {"paid": price, "by": buyer, "for": "one call"}
    return response, receipt

# ...run it on sell_bandwidth, then write your verdict as a comment...

<details><summary>✅ Solution 1 — peek only after trying</summary>

```python
def pay_per_call_v2(service, request, price, buyer):
    pay(buyer, "Provider", price)
    response = service(request)
    if response is None:
        pay("Provider", buyer, price)          # refund
        return None, None
    receipt = {"paid": price, "by": buyer, "for": "one call"}
    return response, receipt

response, receipt = pay_per_call_v2(sell_bandwidth, "50 Mbps, 14:00–16:00",
                                    price=10, buyer="Ada")
print(response)      # "OK — reserved: ..."  → the refund never fires
```

The refund never triggers — a response *exists* (the polite promise), so by the model's
own lights the call succeeded. The failure Ada fears happens at 15:00, long after the
transaction closed. A refund bound to *the response* cannot police a right that
outlives the response. No patch to pay-per-call fixes this; the sold object itself has
to change.

</details>

## 3 · Three ways the naive trade burns (fast-forward)

Strip away even the receipt and watch the raw trade fail. Three robberies at a jog —
chapter 03 slow-cooks each one and builds the machine that stops them.

In [ ]:
# Failure 1 — pay first, get ghosted.
balances = {"Ada": 100, "Bell": 20}
pay("Ada", "Bell", 10)
# Bell configures... nothing. That's the whole attack.
print("balances:", balances, "— Ada is out 10 TOK, owns no bandwidth, and has no recourse.")
print("→ fixed in chapter 03: payment and entitlement swap in one indivisible step.")

In [ ]:
# Failure 2 — serve first, get stiffed.
balances = {"Ada": 100, "Bell": 20}
service_active = True          # Bell provisions first, trusting Ada to pay
# Ada enjoys two hours of guaranteed bandwidth and pays nothing.
print("balances:", balances, "— Bell burned two hours of real capacity for free.")
print("→ also chapter 03: whoever moves first is exposed, so nobody moves first.")

In [ ]:
# Failure 3 — nothing is even provable.
history = [("Ada", "Bell", 10)]                  # a payment log both sides keep... somewhere
history.append(("Bell", "Mallory", 50))          # Mallory forges an entry. Nothing stops this.
print("history:", history)
print("Which entries are real? The data cannot say — anyone can write a tuple.")
print("→ chapter 01: a ledger that can't lie; chapter 04: signatures — proof only the")
print("  author could have produced, checkable by anyone.")

## 4 · The gap: three capabilities, and who has which

Zoom out. This problem sits at the corner of three fields, and systems exist that solve
*pieces* of it. Name the three capabilities precisely (these are the columns of the
paper's Table 1):

- **Machine payment** — a program pays another program, no human clicking "confirm".
- **Token credential** — access is gated by a *transferable token* that is distinct from
  the payment record: an object that can be held, checked, revoked, and change hands.
- **Network activation** — at the end of the pipeline, a real network device actually
  gets configured.

Now score the kinds of systems that exist (described by kind here; the paper does the
citing):

In [ ]:
YES, NO = "✓", "·"

systems = [
    # (system kind,                                        machine-pay, token-cred, net-activation)
    ("pay-per-call for API work (x402-style)",                YES, NO,  NO),
    ("machine purchase of access grants on a ledger",         YES, NO,  NO),
    ("token-gated SDN admission (token from trusted issuer)", NO,  YES, YES),
    ("AI agents operating inside one operator's network",     NO,  NO,  YES),
    ("this project",                                          YES, YES, YES),
]

W = 56
print(f"{'system kind':<{W}} {'machine':>8} {'token':>7} {'network':>9}")
print(f"{'':<{W}} {'payment':>8} {'cred.':>7} {'activ.':>9}")
print("-" * (W + 27))
for name, p, t, a in systems:
    print(f"{name:<{W}} {p:>8} {t:>7} {a:>9}")

Two readings of that table, both load-bearing for the paper:

1. **No earlier row lights all three columns.** Payment systems stop at the receipt;
   the agent-run network systems live *inside* one operator, where there is no
   customer–provider boundary to cross and hence nothing to sell; and the token-gated
   row has real tokens and real devices but no machine payment closing the loop.
2. **The fine print on row three matters most.** In that family, the token is issued by
   a *trusted reservation service* — a middleman you must believe. In this project the
   token is **minted by the settlement itself, atomically against payment, with no
   trusted issuer** — the mint and the payment are one indivisible event (chapter 03 is
   entirely about making that true). Same column checkmark, profoundly different trust
   story.

**✏️ Your turn 2 — score a system you already know**

Add a row for a **cloud provider's API-key subscription** (human signs up with a credit
card, gets an API key, calls a provisioning API). Fill in its three columns and justify
each in a comment. Predict before you print.

In [ ]:
# new_row = ("cloud API-key subscription", ?, ?, ?)
# ...append it to `systems`, reprint the table, justify each column in comments...

<details><summary>✅ Solution 2 — peek only after trying</summary>

```python
systems.append(("cloud API-key subscription", NO, NO, YES))
```

- **Machine payment: ·** — a human onboards with a credit card and a click-through
  contract; the machine only spends against an account a human opened.
- **Token credential: ·** — an API key is a *bearer secret*, not a token-object: it
  can't be held on a neutral registry, transferred, or checked for revocation by anyone
  except the issuer's own backend. (It's precisely the "second trust machine" from the
  🧭 box above.)
- **Network activation: ✓** — the provisioning API does end up configuring real
  infrastructure. That column was never the hard part for clouds; the *boundary
  crossing without a prior relationship* is.

</details>

## 5 · The plan — three questions, eleven notebooks

Everything this course builds is aimed at three research questions — the paper's §3, in
beginner words:

**RQ1 — does the loop actually close?** Can two strangers' agents negotiate, settle
*atomically* (payment and entitlement in one indivisible step), and end with a **real
router** honoring the entitlement, expiring it on time, and killing it early on
revocation — with no human after the initial "Ada, go buy bandwidth"?

**RQ2 — how much of the machinery cares what is being sold?** The project deliberately
sells **two services from different planes**. A quick gloss, because "plane" is
networking jargon: the **dataplane** is the traffic itself — user packets flowing
through the box; the **management plane** is configuring and observing the box. Service
one: a bandwidth cap (dataplane). Service two: a telemetry export — the router streams
its own measurements to a collector (management plane). They could hardly be more
different products. The bet: **everything above one per-service "translator" stays
identical** — same contract, same agents, same checks. If true, a provider adds a third
service by writing one translator. This is the paper's headline claim.

**RQ3 — what does it cost?** Trust minimization isn't free (a chain sits in the loop),
and LLM judgment isn't free (a model thinks for seconds). Measured honestly: what do
they each cost in latency, gas, and tokens — and therefore *what size of deal* is this
architecture practical for?

The road, chapter by chapter:

| chapters | what you build there | which question it serves |
|---|---|---|
| 01 | a ledger that can't lie; keys and addresses; precise data shapes | groundwork |
| 03 | the atomic swap — the settlement vending machine | **RQ1** (settlement half) |
| 04 | signatures a contract can believe (EIP-712) | **RQ1** |
| 05 | the bouncer: ownership proof + the six-check predicate | **RQ1** (enforcement half) |
| 06 | the hands: two translators, two planes, one identical everything-else | **RQ2** |
| 07a·07b | deploy an agent from zero; then this project's two judgment slots | **RQ1** (negotiation) |
| 08 | the whole play, end to end, including mid-window revocation | **RQ1** closed |
| 09 | the evaluation: 80 runs, 12 adversarial probes, the cost of everything | **RQ3** |
| 10 | the paper itself: every claim → its evidence, assembled | all three |

> **🧭 Decision (pragmatic) — two services, one router, one lab**
>
> The prototype sells exactly two services, on one containerized router, on one machine,
> against a local test chain. One *could* demand more before believing anything — many
> devices, many service classes, a public chain, concurrent buyers — and a production
> study should. We didn't, and not because two-on-one is architecturally special:
> **two services from different planes is the smallest experiment that can even ask the
> invariance question (RQ2), and one lab loop is the smallest that can answer RQ1 at
> all.** The paper states this scope plainly in §8.4 (Limitations) and argues in §8.5
> only that the pattern *plausibly* extends — never that it was tested beyond what ran.
> **In the paper:** §5.5 (testbed), §8.4, §8.5.

**✏️ Your turn 3 — retell it**

The oldest test of understanding: say it back. In the cell below, write **one sentence**
(a comment or a string — your choice) that states what this project does, naming all
four trust domains from §1. Then compare against the fold-out — not for wording, for
*coverage*.

In [ ]:
my_retelling = """
...your one sentence here...
"""
print(my_retelling)

<details><summary>✅ Solution 3 — one possible retelling</summary>

> *Ada's agent buys a two-hour bandwidth guarantee from Bell's agent by swapping payment
> for a ticket on a neutral ledger neither of them controls, in one indivisible step —
> and Bell's own gatekeeper then configures the physical routers by trusting nothing
> but that ticket.*

Coverage check: Ada's world (her agent pays and later holds the ticket) ✓ · Bell's world
(his agent sells; his gatekeeper enforces) ✓ · the neutral ground (the ledger where the
swap and the ticket live) ✓ · the network (real routers end up configured) ✓. If your
sentence covered all four, you own the story; wording is yours to keep.

</details>

## 6 · 📝 For the paper

Where this chapter lands (section numbers follow `main-arxiv.tex`, which the full report
expands): the **§1 introduction framing**, the **§3 problem statement and research
questions**, and the **§2.5 gap argument** with its three-column comparison. Draft
sentences you can already defend:

| you can write… | because you ran… |
|---|---|
| *What agents buy over today's machine-payment stack is application-layer work, paid per call: delivery completes with the response, and what persists is a record that value moved.* | §2's pay-per-call simulator on the summarize service — and it was genuinely fine |
| *A network service is the opposite object: when payment settles, everything the buyer paid for still lies ahead — so the sale must leave behind an object naming terms, window, holder, and revocation state, which the seller's own gatekeeper can keep consulting.* | §2's bandwidth purchase at 13:45 and the five unanswerable questions you put to its receipt |
| *Prior systems provide machine payment, or a token credential, or network activation — never all three in one loop; and where a token credential exists, it is issued by a trusted service rather than minted by the settlement itself.* | §4's table — one row lights all three columns, and the fine print on the token-gated row |

**Reviewer objections you can now answer from experience:** *"Why not per-request
payments plus refunds?"* — a refund bound to the response can't police a right that
outlives the response (you built the refund and watched it never fire). *"Why not just
issue an API key after payment?"* — a bearer secret resurrects every trust problem one
layer down; you'd be building a second trust machine to guard the first one's receipt.

**Honesty inventory for §8.4:** two services, one router, one lab machine, a local test
chain — the smallest experiment that can ask RQ1 and RQ2 at all, and exactly as much as
the paper claims.

*Next: [01 — Ledgers, keys, and speaking precisely](01_ledgers_keys_shapes.ipynb) — the
neutral ground has to exist before anything can be swapped on it.*